In [1]:
!pip install pyspark

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, hour, avg

# Create a local Spark cluster simulation
spark = SparkSession.builder \
    .appName("TaxiTipAnalysis") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark Session initialized. Version: {spark.version}")

C:\Users\David Lin\AppData\Roaming\Python\Python314\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark Session initialized. Version: 4.2.0


In [3]:
# Extract the configuration directly from the active Spark Context
#check how many coress PySpark uses
total_cores = spark.sparkContext.defaultParallelism

print(f"🧩 PySpark Architecture Status:")
print(f" - Execution Mode: Local Simulation")
print(f" - Active Distributed Cores Allocated: {total_cores}")
print(f" - Behind the scenes, Spark has divided Taxi dataset into at least {total_cores} parallel partitions!")


🧩 PySpark Architecture Status:
 - Execution Mode: Local Simulation
 - Active Distributed Cores Allocated: 8
 - Behind the scenes, Spark has divided Taxi dataset into at least 8 parallel partitions!


In [ ]:
file_path = "./taxi_data_subset.csv"

# Load the local CSV into a distributed Spark DataFrame
df = spark.read.csv(file_path, header=True, inferSchema=True)

# Verify the upload by printing the row count and schema
print(f"Successfully loaded custom dataset with {df.count():,} rows.")
df.printSchema()


AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/content/taxi_data_subset.csv. SQLSTATE: 42K03

In [ ]:
from pyspark.sql.functions import col, hour, avg, round

# 1. Extract the pickup hour from your custom timestamp column
# 2. Group by hour, calculate average tip, and sort
hourly_tip_df = df.withColumn("pickup_hour", hour(col("tpep_pickup_datetime"))) \
                  .groupBy("pickup_hour") \
                  .agg(round(avg(col("tip_amount")), 2).alias("avg_tip_amount")) \
                  .orderBy("pickup_hour")

# Trigger the action to compute and display your results
hourly_tip_df.show(24)


+-----------+--------------+
|pickup_hour|avg_tip_amount|
+-----------+--------------+
|          0|          3.41|
|          1|          3.35|
|          2|          3.24|
|          3|          3.27|
|          4|          3.11|
|          5|           3.4|
|          6|          3.47|
|          7|           4.1|
|          8|          3.74|
|          9|          3.65|
|         10|          3.35|
|         11|          3.34|
|         12|          3.28|
|         13|          3.31|
|         14|          3.52|
|         15|          3.79|
|         16|          3.94|
|         17|          3.85|
|         23|          0.67|
+-----------+--------------+



In [ ]:

# Print the final execution plan to prove Spark is using distributed DAG processing
#Spark uses lazy evaluation. It means it does not execute code sequentially line-by-line.
#It waits until an action is triggered and works backward to build, optimize, and organize the steps into a logical roadmap.
#When  df.explain() is called, Spark reads from the bottom up (or inside out)
#to display how data flows from the storage disk into the final output.
hourly_tip_df.explain()

# Trigger action to compute and display results
hourly_tip_df.show(24)


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [pickup_hour#62 ASC NULLS FIRST], true, 0
   +- Exchange rangepartitioning(pickup_hour#62 ASC NULLS FIRST, 200), ENSURE_REQUIREMENTS, [plan_id=130]
      +- HashAggregate(keys=[pickup_hour#62], functions=[avg(tip_amount#30)])
         +- Exchange hashpartitioning(pickup_hour#62, 200), ENSURE_REQUIREMENTS, [plan_id=127]
            +- HashAggregate(keys=[pickup_hour#62], functions=[partial_avg(tip_amount#30)])
               +- Project [tip_amount#30, hour(tpep_pickup_datetime#18, Some(Etc/UTC)) AS pickup_hour#62]
                  +- FileScan csv [tpep_pickup_datetime#18,tip_amount#30] Batched: false, DataFilters: [], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/content/taxi_data_subset.csv], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<tpep_pickup_datetime:timestamp,tip_amount:double>


+-----------+--------------+
|pickup_hour|avg_tip_amount|
+-----------+--------------+
|          0|          3